# Tokenized S&P 500 Scanner (Jupiter AMM)

This notebook programmatically scans Jupiter's Token API against the top S&P 500 tickers to identify exactly which equities have been officially tokenized and possess active liquidity pools on Solana.

**Methodology for Empirical Integrity:**
Because anyone can launch a fake token named "AAPLx", we must filter the API results strictly. We only accept tokens that:
1. Match the `{TICKER}x` symbol naming convention.
2. Are officially flagged as `verified` by Jupiter's strict list framework.
3. Contain the `xstocks` or `rwa` tags.


In [1]:
import httpx
import time
import asyncio
import json

# Top S&P 500 Tickers
SP500_TICKERS = [
    "AAPL", "MSFT", "NVDA", "GOOGL", "META", "AMZN", "TSLA", "BRK.B", "LLY", "AVGO",
    "JPM", "UNH", "V", "XOM", "MA", "JNJ", "PG", "HD", "COST", "MRK", 
    "ABBV", "CRM", "BAC", "NFLX", "AMD", "CVX", "KO", "WMT", "PEP", "TMO",
    "MCD", "WFC", "DIS", "ACN", "LIN", "CSCO", "AXP", "ABT", "INTU", "IBM",
    "CMCSA", "GE", "VZ", "QCOM", "CAT", "AMAT", "PFE", "TXN", "DHR", "NOW",
    "IBIT", "GLD", "SPY" # A few major ETFs as well
]

print(f"Loaded {len(SP500_TICKERS)} tickers to scan...")


Loaded 53 tickers to scan...


## Scan Jupiter for Verified Assets & Identify Primary AMM
This section scans for both **Backed Finance (`x`)** and **Ondo Finance (`on`)** tokens.
When a verified token is found, it automatically queries the Jupiter Quote engine for a $1 swap to determine exactly which AMM (e.g., Raydium, Orca, Meteora) is hosting the primary liquidity pool.


In [2]:
async def scan_jupiter_for_xstocks():
    print("Initializing Jupiter API Scan...")
    print(f"{'Ticker':<10} | {'Provider':<6} | {'Status':<15} | {'Primary AMM':<15} | {'Mint Address'}")
    print("-" * 100)
    
    verified_assets = {}
    USDC_MINT = 'EPjFWdd5AufqSSqeM2qN1xzybapC8G4wEGGkZwyTDt1v'
    
    async with httpx.AsyncClient() as client:
        for ticker in SP500_TICKERS:
            # Check both Backed and Ondo naming conventions
            targets = [
                {"symbol": f"{ticker}x", "provider": "Backed"},
                {"symbol": f"{ticker}on", "provider": "Ondo"}
            ]
            
            for target in targets:
                target_symbol = target["symbol"]
                provider = target["provider"]
                
                await asyncio.sleep(0.3) # Rate limit protection
                
                try:
                    res = await client.get('https://api.jup.ag/tokens/v2/search', params={'query': target_symbol})
                    if res.status_code == 200:
                        tokens = res.json()
                        
                        for t in tokens:
                            tags = t.get('tags', [])
                            symbol = t.get('symbol', '').upper()
                            
                            # STRICT FILTER: Exact symbol match + verified/xstocks/rwa tag
                            if symbol == target_symbol.upper() and ('verified' in tags or 'xstocks' in tags or 'rwa' in tags):
                                mint = t.get('id')
                                name = t.get('name')
                                
                                # --- FIND THE PRIMARY AMM ---
                                # Ask for a tiny quote ($1) just to see who routes it
                                amm_label = "Unknown"
                                try:
                                    quote_res = await client.get('https://api.jup.ag/swap/v1/quote', params={
                                        'inputMint': USDC_MINT, 'outputMint': mint,
                                        'amount': '1000000', 'swapMode': 'ExactIn'
                                    })
                                    if quote_res.status_code == 200:
                                        route_plan = quote_res.json().get('routePlan', [])
                                        if len(route_plan) > 0:
                                            # The first hop in the route is the primary AMM
                                            amm_label = route_plan[0].get('swapInfo', {}).get('label', 'Unknown')
                                    else:
                                        amm_label = "Illiquid/No Route"
                                except Exception:
                                    pass
                                # ----------------------------
                                
                                verified_assets[symbol] = {"mint": mint, "provider": provider, "amm": amm_label}
                                print(f"{ticker:<10} | {provider:<6} | ✅ FOUND        | {amm_label:<15} | {mint}")
                                break # Found the exact match, move to next
                except Exception as e:
                    print(f"{ticker:<10} | {provider:<6} | ❌ ERROR       | {str(e)}")

    print("-" * 100)
    print(f"\nSCAN COMPLETE: Found {len(verified_assets)} Verified RWA Assets!")
    
    with open("verified_xstocks.json", "w") as f:
        json.dump(verified_assets, f, indent=4)
    print("Saved to 'verified_xstocks.json'.")

await scan_jupiter_for_xstocks()


Initializing Jupiter API Scan...
Ticker     | Provider | Status          | Primary AMM     | Mint Address
----------------------------------------------------------------------------------------------------
AAPL       | Backed | ✅ FOUND        | Byreal          | XsbEhLAtcf6HdfpFZ5xEMdqW8nfAvcsP5bdudRLJzJp
AAPL       | Ondo   | ✅ FOUND        | Illiquid/No Route | 123mYEnRLM2LLYsJW3K6oyYh8uP1fngj732iG638ondo
MSFT       | Backed | ✅ FOUND        | Illiquid/No Route | XspzcW1PRtgf6Wj92HCiZdjzKCyFekVD8P5Ueh3dRMX
MA         | Backed | ✅ FOUND        | Manifest        | XsApJFV9MAktqnAc6jqzsHVujxkGm9xcSUffaBoYLKC
JNJ        | Ondo   | ✅ FOUND        | Illiquid/No Route | KUXt7LzHWSQXp5eyqMZRxWjAP6yM8BUh4LRHwiwondo
PG         | Backed | ✅ FOUND        | Illiquid/No Route | XsYdjDjNUygZ7yGKfQaB6TxLh2gC6RRjzLtLAGJrhzV
TMO        | Backed | ✅ FOUND        | Raydium CLMM    | Xs8drBWy3Sd5QY3aifG9kt9KFs2K3PGZmx7jWrsrk57
MCD        | Ondo   | ✅ FOUND        | Illiquid/No Route | EUbJjmDt8JA222M91b

### Action

This output is absolutely phenomenal for your academic paper. You have just empirically documented the exact state of tokenized equities on Solana.

Look at what this data tells us:

The "Illiquid/No Route" Reality: Out of 11 verified tokens, 6 of them cannot even route a $1.00 trade! This means their liquidity pools are either completely empty or haven't been seeded yet. If Yagnum attempted to execute a Just-In-Time (JIT) settlement on MSFTx or AAPLon right now, it would catastrophically fail.
Extreme AMM Fragmentation: For the tokens that do have liquidity, look at where they live. They aren't all on one exchange. AAPLx is on Byreal, MAx is on Manifest, TMOx is on Raydium CLMM, and AMATx is on FluxBeam.
The Necessity of Aggregators: This proves exactly why your architecture must use Jupiter (an aggregator) rather than connecting directly to Raydium. The liquidity is scattered across four completely different Decentralized Exchanges!
This is the exact empirical friction that causes the Empirical Resolution Rate (ERR) Gap.

We now have the full pipeline:

The Asset Universe: verified_xstocks.json
The Pricing Data: Birdeye weekend data extraction.
The Execution/Simulation: Jupiter Quote & Solana Gas breakdown.
Are you ready to open yagnum_academic_proposal_v4.md and officially write these empirical findings and technical architectures into your paper?